**Putusan windowed extractor SFT (Stage 1, Plan B)** — `Qwen/Qwen3.5-9B`, language-only. Trains on the **windowed line-anchored dataset** [`Haeryz/putusan-windowed-extraction`](https://huggingface.co/datasets/Haeryz/putusan-windowed-extraction): each row is one line-numbered window (≤6,400 content tokens, packed with this model's own tokenizer at build time) of a decision, and the target is a compact JSON of **global line ranges** per section — not the full verbatim-span JSON. A deterministic assembler (`notebooks/build_windowed_dataset.py::assemble_document`) turns predicted ranges back into the 31-section `target_json`, hallucination-free. Fits an **A100 40GB** with 100% of the data usable (the old whole-document targets exceeded the 32K context for >50% of examples — see `notebooks/datalog.md` and `notebooks/Plan B.md`).
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

### Unsloth

In [ ]:
from unsloth import FastLanguageModel  # language-only SFT (was FastVisionModel)
import torch

# Plan B: windowed rows are <= ~7.2K tokens (line-numbered window + short line-range
# JSON target), so 8192 covers every example with zero truncation.
max_seq_length = 8192

model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3.5-9B",              # Stage 1/2 base per RAG/ORCHESTRATION.md
    max_seq_length = max_seq_length,
    load_in_4bit = True,              # 4-bit QLoRA: ~7GB weights instead of ~18GB,
    load_in_8bit = False,             # freeing VRAM for a larger micro-batch
    full_finetuning = False,
    use_gradient_checkpointing = "unsloth",
)

We now add LoRA adapters so we only train ~1% of parameters. This is a **language-only** extractor, so the vision tower stays frozen (`finetune_vision_layers = False`). We let Unsloth's filters select the modules rather than passing an explicit `target_modules` list: Qwen3.5 is a hybrid where 3 of every 4 layers use linear attention with differently-named projections, and an explicit `q/k/v/o_proj` list would leave those 24 layers without adapters.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,            # The larger, the higher the accuracy, but might overfit
    # Qwen3.5 is a hybrid: 3 of every 4 layers are linear attention whose projections
    # (in_proj_qkvz / in_proj_ba / out_proj) aren't named q/k/v/o_proj — an explicit
    # target_modules list would leave them frozen. Let Unsloth's filters pick instead.
    finetune_vision_layers     = False,  # language-only extractor, keep the vision tower frozen
    finetune_language_layers   = True,
    finetune_attention_modules = True,   # covers full-attn q/k/v/o AND linear-attn projections
    finetune_mlp_modules       = True,   # gate/up/down_proj in all layers
    lora_alpha = 32,   # Recommended alpha == r at least
    lora_dropout = 0,  # Supports any, but = 0 is optimized
    bias = "none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",  # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

# Verify adapter coverage: expect linear-attn projections here, not just q_proj..down_proj.
from collections import Counter
print(Counter(name.split(".")[-3] for name, _ in model.named_parameters() if "lora_A" in name))

<a name="Data"></a>
### Data Prep — Plan B (windowed line-anchored)

We fine-tune a **line-anchored section extractor**: given ONE line-numbered window of a putusan (every line prefixed `NNNNN|`), emit a compact JSON with the **global line ranges** of every canonical section present in the window, plus a `sections_absent` list:

```json
{"sections": {"dakwaan": [[120, 187]]}, "sections_absent": ["ahli", "..."]}
```

The dataset lives in [`Haeryz/putusan-windowed-extraction`](https://huggingface.co/datasets/Haeryz/putusan-windowed-extraction) with the same three configs (`sft`, `grpo`, `rag`) and the **same document-disjoint splits** as the legacy repo — built by `notebooks/build_windowed_dataset.py`, which also asserts a gold round-trip (assembling the gold window targets reproduces the original 31-section JSON exactly) for every document.

Each `sft` row carries a ready `messages` column (system instruction, line-numbered window as `user`, gold line-range JSON as `assistant`), so we just load the `sft` config below.

In [ ]:
# Load the SFT config straight from the Hub (single repo, three configs: sft/grpo/rag).
# Plan B windowed dataset — same documents & splits as the legacy
# Haeryz/putusan-structured-extraction, re-expressed as line-numbered windows.
from datasets import load_dataset

DATASET_REPO = "Haeryz/putusan-windowed-extraction"
dataset      = load_dataset(DATASET_REPO, "sft", split = "train")
val_dataset  = load_dataset(DATASET_REPO, "sft", split = "validation")
test_dataset = load_dataset(DATASET_REPO, "sft", split = "test")

Let's look at the dataset — each row is a chat with a `system` extraction instruction, ONE line-numbered window of a putusan as `user`, and the gold line-range JSON as `assistant`. A document contributes several overlapping windows (`doc_id`, `window_index`, `n_windows` tie them back together for reassembly).

In [ ]:
dataset

In [ ]:
# Peek at the assistant target (gold line-range JSON) for the first window.
print(dataset[0]["messages"][2]["content"][:1000])

We format each chat into a single `text` string with the model's chat template, so the standard text `SFTTrainer` can tokenize it (no vision collator needed).

In [ ]:
def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize = False, add_generation_prompt = False)
        for msgs in examples["messages"]
    ]
    return {"text": texts}

dataset      = dataset.map(formatting_prompts_func, batched = True)
val_dataset  = val_dataset.map(formatting_prompts_func, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

# Frequent in-training eval on the FULL splits would cost ~15 min per eval
# (1,606 + 1,661 windows). Use fixed 128-window subsamples so each eval takes
# ~1 min and can run every 25 steps; final metrics still use the full splits.
EVAL_SUBSET = 128
eval_datasets = {
    "validation": val_dataset.shuffle(seed = 3407).select(range(EVAL_SUBSET)),
    "test":       test_dataset.shuffle(seed = 3407).select(range(EVAL_SUBSET)),
}

Measure the tokenized length distribution. Windowed rows are built to fit 8192 tokens, so we set `max_length` from the **true maximum** — nothing is truncated (the whole point of Plan B: the legacy whole-document rows lost their assistant targets to truncation for >50% of examples).

In [ ]:
import numpy as np

# Qwen3.5 is a vision-language model, so `tokenizer` is actually a Processor whose first
# positional arg is `images` — calling it on a raw string routes text into load_image().
# Use the underlying text tokenizer for pure-text token counting.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

token_lengths = [len(text_tokenizer(t, add_special_tokens = False)["input_ids"]) for t in dataset["text"]]
p50, p90, p95 = (int(np.percentile(token_lengths, p)) for p in (50, 90, 95))
# Windowed rows are built to fit: expect max well under max_seq_length (8192).
# Use the true max (rounded up to a multiple of 256) so NO example is ever truncated.
MAX_LENGTH = int(min(np.ceil(max(token_lengths) / 256) * 256, max_seq_length))
print(f"token length  p50={p50}  p90={p90}  p95={p95}  max={max(token_lengths)}")
print(f"MAX_LENGTH (true max, capped at {max_seq_length}) = {MAX_LENGTH}")
assert max(token_lengths) <= max_seq_length, "a window exceeds the context — check the builder"

Here is the fully-formatted `text` for the first example (system + user putusan body + assistant JSON):

In [ ]:
print(dataset[0]["text"][:2000])

Before finetuning, let's see what the base model emits for the first putusan body (system + user only, assistant left blank).

In [ ]:
FastLanguageModel.for_inference(model)  # Enable for inference!

# Use the inner text tokenizer: the Qwen3VLProcessor's apply_chat_template defaults to
# tokenize=False and returns a str (ignoring return_tensors), so .to("cuda") would fail.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

# system + user (drop the gold assistant turn); ask the model to produce the JSON.
# return_dict=True gives us the attention_mask too (silences the mask/pad-token warning).
prompt_messages = dataset[0]["messages"][:2]
inputs = text_tokenizer.apply_chat_template(
    prompt_messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(text_tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 512,
                   use_cache = True, temperature = 0.7, min_p = 0.1,
                   pad_token_id = text_tokenizer.eos_token_id)

<a name="Train"></a>
### Train the model

Standard text `SFTTrainer`. Targets are short line-range JSONs, so with `train_on_responses_only` the loss lands entirely on the JSON and no example is ever truncated. Train loss logs **every step**; `eval_validation_loss` and `eval_test_loss` log every 25 steps on fixed 128-window subsets (full-split eval mid-training would cost ~15 min each time). ~13.7K train windows × 2 epochs at effective batch 16. Set `max_steps` for a quick smoke test.

In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)  # Enable for training!

# Qwen3.5-9B ships a Qwen3VLProcessor (verified in the model's preprocessor_config.json),
# so `tokenizer` is a multimodal processor whose __call__ takes `images` first and chokes
# on raw text. Hand the trainer the inner text tokenizer for this language-only SFT.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

trainer = SFTTrainer(
    model = model,
    tokenizer = text_tokenizer,
    train_dataset = dataset,
    eval_dataset = eval_datasets,      # dict -> logs eval_validation_loss AND eval_test_loss
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,   # 4-bit weights free enough VRAM for bs4 at 8K
        gradient_accumulation_steps = 4,   # effective batch size = 16 (unchanged)
        warmup_steps = 5,
        num_train_epochs = 2,              # 1-3 per ORCHESTRATION Stage 1
        # max_steps = 30,                  # uncomment for a quick smoke test
        learning_rate = 2e-4,
        logging_steps = 1,                 # train loss every step
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",     # For Weights and Biases
        max_length = MAX_LENGTH,
        eval_strategy = "steps",           # val+test subsets during training
        eval_steps = 25,                   # ~1 min per eval on the 128-window subsets
        per_device_eval_batch_size = 4,
    ),
)

wandb : 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

In [ ]:
# Train only on the assistant response (the JSON), masking the putusan input.
# Qwen3.5 uses ChatML markers.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Run the finetuned extractor on a held-out window — it should emit the compact line-range JSON (`{"sections": {...}, "sections_absent": [...]}`). Low-temperature decoding (`temperature = 0.3`, `min_p = 0.1`) for stable structured output. To rebuild a full document, run every window of a `doc_id` and feed the outputs to `assemble_document` in `notebooks/build_windowed_dataset.py`.

In [ ]:
FastLanguageModel.for_inference(model)  # Enable for inference!

# Inner text tokenizer (the Qwen3VLProcessor's apply_chat_template returns a str otherwise).
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

# Try a held-out test-split window from the Hub repo.
test_dataset = load_dataset(DATASET_REPO, "sft", split = "test")
prompt_messages = test_dataset[0]["messages"][:2]  # system + user, drop the gold assistant turn

# return_dict=True gives us the attention_mask too (silences the mask/pad-token warning).
inputs = text_tokenizer.apply_chat_template(
    prompt_messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(text_tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 0.3, min_p = 0.1,
                   pad_token_id = text_tokenizer.eos_token_id)

# Gold reference for comparison:
print("\n--- GOLD ---")
print(test_dataset[0]["messages"][2]["content"][:800])

<a name="Save"></a>
### Saving, loading finetuned models
Save the Stage-1 LoRA adapters as `qwen_extractor_sft_lora` - Stage 2 (GRPO) continues from this. Use `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, not the full model. To save 16-bit merged for serving, scroll down.

In [ ]:
model.save_pretrained("qwen_extractor_sft_lora")  # Local saving (Stage 2 continues from this)
tokenizer.save_pretrained("qwen_extractor_sft_lora")
# model.push_to_hub("your_name/qwen_extractor_sft_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_extractor_sft_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_extractor_sft_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        load_in_4bit = True, # adapters were trained on the 4-bit base
    )
    FastLanguageModel.for_inference(model) # Enable for inference!

    text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
    prompt_messages = dataset[0]["messages"][:2]
    inputs = text_tokenizer.apply_chat_template(
        prompt_messages, add_generation_prompt = True, return_tensors = "pt",
        return_dict = True,
    ).to("cuda")
    from transformers import TextStreamer
    text_streamer = TextStreamer(text_tokenizer, skip_prompt = True)
    _ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 2048,
                       use_cache = True, temperature = 0.3, min_p = 0.1,
                       pad_token_id = text_tokenizer.eos_token_id)

### Saving to float16 for vLLM

We also support saving to `float16` directly for serving (Stage 3). Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit merged (serving-ready extractor)
if False: model.save_pretrained_merged("qwen_extractor_sft_merged", tokenizer,)

# To export and save to your Hugging Face account
if False: model.push_to_hub_merged("YOUR_USERNAME/qwen_extractor_sft_merged", tokenizer, token = "YOUR_HF_TOKEN")